<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/solved/12_robust_student_t_and_loo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 12 — Robust likelihood and predictive comparison

Return to the Gaussian hierarchical models, use leave-one-out predictive checks to expose influential observations, and compare them with Student-t observation models.

## Setup

This course pins PyMC, modular ArviZ, and Bambi for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pandas==2.2.3" \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1" \
    "bambi==0.21.0"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bambi as bmb
import pymc as pm
import arviz_base as azb
import arviz_stats as azs
import arviz_plots as azp

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("Bambi:", bmb.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

That zero point is scientifically meaningful, so every Bambi model in this sequence uses `center_predictors=False`. The `Intercept` prior is therefore a prior on baseline reaction time rather than reaction time at the average deprivation day.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

print(f"{sleep['Subject'].nunique()} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

# 12.1 Difficult-to-predict observations

Do Gaussian hierarchical models contain observations that are difficult to predict when they are left out of the fit?

We refit four standalone models here so the notebook does not depend on state from earlier notebooks: Gaussian and Student-t likelihoods, each with varying intercepts only or varying intercepts plus slopes. Bambi's group-specific effects are independent, as in notebooks 5–11.

In [ ]:
base_vi_priors = {
    "Intercept": bmb.Prior("Normal", mu=250, sigma=100),
    "Days": bmb.Prior("Normal", mu=0, sigma=20),
    "sigma": bmb.Prior("Exponential", lam=0.04),
    "1|Subject": bmb.Prior(
        "Normal", mu=0, sigma=bmb.Prior("Exponential", lam=0.04)
    ),
}
base_vis_priors = {
    **base_vi_priors,
    "Days|Subject": bmb.Prior(
        "Normal", mu=0, sigma=bmb.Prior("Exponential", lam=0.10)
    ),
}

models = {
    "gaussian_intercept": bmb.Model(
        "Reaction ~ Days + (1 | Subject)", sleep, family="gaussian",
        priors=base_vi_priors, categorical="Subject", center_predictors=False,
    ),
    "gaussian_slope": bmb.Model(
        "Reaction ~ Days + (1 + Days | Subject)", sleep, family="gaussian",
        priors=base_vis_priors, categorical="Subject", center_predictors=False,
    ),
    "student_t_intercept": bmb.Model(
        "Reaction ~ Days + (1 | Subject)", sleep, family="t",
        priors=base_vi_priors, categorical="Subject", center_predictors=False,
    ),
    "student_t_slope": bmb.Model(
        "Reaction ~ Days + (1 + Days | Subject)", sleep, family="t",
        priors=base_vis_priors, categorical="Subject", center_predictors=False,
    ),
}
models["student_t_slope"]

In [ ]:
idatas = {}
for name, comparison_model in models.items():
    print("\n---", name, "---")
    idatas[name] = comparison_model.fit(
        draws=1000, tune=1500, chains=4, target_accept=0.95, random_seed=RANDOM_SEED
    )
    print("Divergences:", int(idatas[name]["sample_stats"]["diverging"].sum().item()))
    comparison_model.predict(
        idatas[name], kind="response", inplace=True, random_seed=RANDOM_SEED
    )
    comparison_model.compute_log_likelihood(idatas[name])

These intervals use PSIS-LOO weighting rather than the ordinary full-data posterior predictive distribution. Large discrepancies help identify observations for which the fitted model depends strongly on seeing that observation.

In [ ]:
azp.plot_loo_interval(
    idatas["gaussian_slope"], var_names=["Reaction"],
    point_estimate="mean", ci_probs=(0.50, 0.90), ci_kind="hdi",
    figure_kwargs={"figsize": (11, 4)},
);

In [ ]:
azp.plot_loo_interval(
    idatas["student_t_slope"], var_names=["Reaction"],
    point_estimate="mean", ci_probs=(0.50, 0.90), ci_kind="hdi",
    figure_kwargs={"figsize": (11, 4)},
);

# 12.2 Robust predictive calibration

Does replacing the Gaussian likelihood with a Student-t likelihood improve leave-one-out calibration?

In [ ]:
azp.plot_loo_pit(idatas["gaussian_slope"], var_names=["Reaction"]);
azp.plot_loo_pit(idatas["student_t_slope"], var_names=["Reaction"]);

# 12.3 Predictive comparison

What do PSIS-LOO diagnostics and ELPD differences say about likelihood choice and the value of varying slopes?

In [ ]:
loos = {
    name: azs.loo(idata, var_name="Reaction", pointwise=True)
    for name, idata in idatas.items()
}

for name, loo in loos.items():
    print(name, "ELPD:", round(float(loo.elpd), 2), "SE:", round(float(loo.se), 2),
          "max Pareto k:", round(float(loo.pareto_k.max()), 2))

comparison = azs.compare(loos, method="stacking", round_to="none")
comparison

In [ ]:
azp.plot_compare(comparison);

In [ ]:
azp.plot_khat(loos["gaussian_slope"]);
azp.plot_khat(loos["student_t_slope"]);

# 12.4 Prediction target

What prediction problem does this leave-one-observation-out comparison actually answer?

This comparison asks about prediction of another observation from an **already observed participant**, because rows—not whole participants—are left out. It is not a validation of prediction for a completely new participant. Compare ELPD differences together with their uncertainty and Pareto-$k$ diagnostics; do not treat a numerical rank as proof that one scientific model is true.